In [1]:
from langchain.document_loaders import PyMuPDFLoader, TextLoader, Docx2txtLoader, UnstructuredWordDocumentLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter 
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from autogen_ext.memory.chromadb import ChromaDBVectorMemory, PersistentChromaDBVectorMemoryConfig
from autogen_core.memory import MemoryContent, MemoryMimeType
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
import logging
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
import os

In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_core.models import ModelInfo

model_info = ModelInfo(
    model_name="dolphin-2.9.3-mistral-nemo-12b",
    function_calling=True,  # Indica si el modelo soporta llamadas a funciones
    vision=False,           # Indica si el modelo tiene capacidades de visión
    json_output=True,      # Indica si el modelo puede generar salida en formato JSON
    family="unknown",       # Specify the model family if known
    structured_output=False
) # type: ignore

# Define a model client. You can use other model client that implements
# the `ChatCompletionClient` interface.

#Model client for LMStudio
model_client = OpenAIChatCompletionClient(
    model="dolphin-2.9.3-mistral-nemo-12b",
    base_url="http://127.0.0.1:1234/v1",
    api_key="LMStudio", 
    model_info=model_info,
)

#Model client for Ollama
ollama_model_client = OllamaChatCompletionClient(
    model="qwen2.5:14b", 
    host="https://ollama01.decoupled.ai", 
    model_info=model_info)

In [3]:
### --- CONFIG PATH--- ###
CHROMA_DIR = "../chromadb_storage"
DOCS_DIR = "../data"
COLLECTION_NAME = "rag_collection"

### --- EMBEDDING FUNCTION (Local SentenceTransformer) --- ###
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

/home/jun/Documents/Master_IA/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Step 2: Set up ChromaDB client with embedding function
chroma_client = chromadb.Client(Settings(
    persist_directory=CHROMA_DIR,
    anonymized_telemetry=False
))

# Step 3: Create or load the collection WITH embedding
collections = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

In [5]:
### --- RAG-ENABLED AUTO-GEN SETUP WITH CHROMADB --- ###

vector_memory = ChromaDBVectorMemory(
    config=PersistentChromaDBVectorMemoryConfig(
        collection_name=COLLECTION_NAME,
        persistence_path=CHROMA_DIR,
        k=3,
        score_threshold=0.4,
    )
)

In [6]:
#Funtion to load and index documents
# This function loads documents from a specified directory, splits them into chunks,
# and indexes them in the provided vector memory.
# It supports PDF, TXT, and DOCX formats.
# The function uses a text splitter to divide the documents into smaller chunks
# based on the specified chunk size and overlap.
# It also handles exceptions for unsupported file formats and errors during processing.

async def load_and_index_documents(docs_dir, vector_memory, chunk_size=500, chunk_overlap=50):
    """
    Loads PDF, TXT, and DOCX documents from the specified directory,
    splits them into chunks, and indexes them in the provided AutoGen vector memory.

    Args:
        docs_dir (str): Path to the directory containing the documents.
        vector_memory (ChromaDBVectorMemory): The vector memory object to index the documents into.
        chunk_size (int): Number of characters in each chunk.
        chunk_overlap (int): Number of overlapping characters between chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    for file_name in os.listdir(docs_dir):
        file_path = os.path.join(docs_dir, file_name)
        ext = os.path.splitext(file_name)[-1].lower()

        try:
            if ext == ".pdf":
                loader = PyMuPDFLoader(file_path)
            elif ext == ".txt":
                loader = TextLoader(file_path)
            elif ext == ".docx":
                loader = UnstructuredWordDocumentLoader(file_path)
            else:
                print(f"Unsupported file format: {file_name}")
                continue

            docs = loader.load()
            split_docs = text_splitter.split_documents(docs)

            for i, chunk in enumerate(split_docs):
                await vector_memory.add(  # Add `await` here
                    MemoryContent(
                        content=chunk.page_content,
                        mime_type=MemoryMimeType.TEXT,
                        metadata={"source": file_name, "chunk_index": i}
                    )
                )

            print(f"Indexed {len(split_docs)} chunks from {file_name}.")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")
    return chunk_size, chunk_overlap

In [7]:
# Load and index documents
await vector_memory.clear()
await load_and_index_documents(DOCS_DIR, vector_memory)

Indexed 1313 chunks from clean-architecture.pdf.
Indexed 2197 chunks from clean-code.pdf.


(500, 50)

In [13]:
#Set up the Rag-enabled AutoGen agent and the user proxy
# The user proxy is a special agent that represents the human user in the conversation.
# It can be used to send messages to the assistant agent and receive responses.
# The assistant agent is the main agent that interacts with the user and performs tasks.

user_proxy = UserProxyAgent(
    name="user_proxy",
    description="A human user",
    input_func=input
)

assistant_agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    memory=[vector_memory],
    system_message="You are a helpful assistant that use RAG to answer questions"  # if you have a custom system prompt
)


In [14]:
#Set up the termination word for the conversation
# The termination condition is a special condition that indicates when the conversation should end.
# In this case, the conversation will end when the user types "exit".
termination = TextMentionTermination("exit")

# Create a round-robin group chat team with both agents
team = RoundRobinGroupChat(
    [assistant_agent, user_proxy], 
    termination_condition=termination
)

In [ ]:
# stream = assistant_agent.run_stream(task="what are the  books that all serious practitioners will have on their bookshelves?")
# await Console(stream)

# await vector_memory.close()
# await model_client.close()

---------- user ----------
what are the  books that all serious practitioners will have on their bookshelves?
---------- assistant ----------
[MemoryContent(content='about programming, there will be lots of code. If the book is about managing, there \nwill be lots of case studies from real projects. \nThese are the books that all serious practitioners will have on their bookshelves. \nThese are the books that will be remembered for making a difference and for guiding \nprofessionals to become true craftsman. \nManaging Agile Projects\nSanjiv Augustine\nAgile Estimating and Planning\nMike Cohn\nWorking Effectively with Legacy Code\nMichael C. Feathers', mime_type='MemoryMimeType.TEXT', metadata={'chunk_index': 3, 'mime_type': 'MemoryMimeType.TEXT', 'source': 'clean-code.pdf', 'score': 0.5309425592422485, 'id': '526b59c4-e355-42f4-b3f7-fbcb55ccec6f'}), MemoryContent(content='making readable, 311\nnecessity of, 2\nreading from top to bottom, 37\nsimplicity of, 18, 19\ntechnique for shroud

In [16]:
# Function to run the chat

initial_question = "I have a question about clean code. Can you help me?"
stream = team.run_stream(task=initial_question)
await Console(stream)  # stream the conversation to console

---------- user ----------
I have a question about clean code. Can you help me?
---------- assistant ----------
[MemoryContent(content='In short, a programmer who writes clean code is an artist who can take a blank screen\nthrough a series of transformations until it is an elegantly coded system.\nWhat Is Clean Code?\nThere are probably as many deﬁnitions as there are programmers. So I asked some very\nwell-known and deeply experienced programmers what they thought. \nBjarne Stroustrup, inventor of C++ \nand author of The C++ Programming \nLanguage\nI like my code to be elegant and efﬁcient. The', mime_type='MemoryMimeType.TEXT', metadata={'chunk_index': 183, 'mime_type': 'MemoryMimeType.TEXT', 'source': 'clean-code.pdf', 'score': 0.6986156404018402, 'id': 'f6b7fc0a-c609-4754-911c-0dbb42b01fd2'}), MemoryContent(content='2\nChapter 1: Clean Code\nThis is a book about good programming. It is ﬁlled with code. We are going to look at\ncode from every different direction. We’ll look down at

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='I have a question about clean code. Can you help me?', type='TextMessage'), MemoryQueryEvent(source='assistant', models_usage=None, metadata={}, content=[MemoryContent(content='In short, a programmer who writes clean code is an artist who can take a blank screen\nthrough a series of transformations until it is an elegantly coded system.\nWhat Is Clean Code?\nThere are probably as many deﬁnitions as there are programmers. So I asked some very\nwell-known and deeply experienced programmers what they thought. \nBjarne Stroustrup, inventor of C++ \nand author of The C++ Programming \nLanguage\nI like my code to be elegant and efﬁcient. The', mime_type='MemoryMimeType.TEXT', metadata={'chunk_index': 183, 'mime_type': 'MemoryMimeType.TEXT', 'source': 'clean-code.pdf', 'score': 0.6986156404018402, 'id': 'f6b7fc0a-c609-4754-911c-0dbb42b01fd2'}), MemoryContent(content='2\nChapter 1: Clean Code\nThis is a bo

In [24]:
await vector_memory.close()
await model_client.close()

In [14]:
# Adjust the path to your DB directory
client = chromadb.PersistentClient(path=CHROMA_DIR)

# List all collections
collections = client.list_collections()
print(collections)

# Example: inspect one collection
collection = client.get_collection(COLLECTION_NAME)

# Fetch all documents (or a subset)
results = collection.get(limit=5)  # Adjust limit as needed
print("IDs:", results['ids'])
print("Documents:", results['documents'])  # Optional, if you stored them
print("Metadatas:", results['metadatas'])  # Optional
print("Embeddings:", results['embeddings'])  # Optional

['rag_collection']
IDs: ['2ddfb6f4-b2d8-4d7e-81fb-2ae83094b054', 'c0d22b50-424b-497f-9724-0f0cfb42a226', 'f133d678-280f-4350-a33b-8e43fa8309cb', 'aba116de-45bb-4e81-92e0-9846c3376044', '0339b791-e648-4476-a9cb-4a089e4f3654']
Documents: ['About This E-Book\nEPUB is an open, industry-standard format for e-books. However, support for\nEPUB and its many features varies across reading devices and applications. Use\nyour device or app settings to customize the presentation to your liking. Settings\nthat you can customize often include font, font size, single or double column,\nlandscape or portrait mode, and figures that you can click or tap to enlarge. For\nadditional information about the settings and features on your reading device or', 'app, visit the device manufacturer’s Web site.\nMany titles include programming code or configuration examples. To optimize\nthe presentation of these elements, view the e-book in single-column, landscape\nmode and adjust the font size to the smallest set